In [20]:
import getpass
import os


def set_api_key(key_name: str) -> None:
    """
    Securely set an environment variable if it doesn't already exist.
    Prompts the user for input using a password-style hidden input.
    
    Args:
        key_name (str): Name of the environment variable to set (e.g., "OPENAI_API_KEY")
    """
    if not os.environ.get(key_name):
        os.environ[key_name] = getpass.getpass(f"{key_name}: ")

# Example usage:
set_api_key("OPENAI_API_KEY")
# set_api_key("ANTHROPIC_API_KEY")

In [21]:
os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")
set_api_key("LANGCHAIN_API_KEY")

In [22]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

In [23]:
len(docs)

64

In [24]:
from langchain_openai import ChatOpenAI
from ragas.llms import LangchainLLMWrapper
from langchain_openai import OpenAIEmbeddings
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy
from sqlalchemy import true

  # LM Studio uses OpenAI-compatible API format
generator_llm = ChatOpenAI(
      base_url="http://192.168.1.157:1234/v1",  # Replace with your server 
      api_key="lmstudio",  # LM Studio doesn't require real API key
      model="openai/gpt-oss-120b",  # Optional: specify model name
      temperature=0.7,
      verbose=True 
  )

try:
  response = generator_llm.invoke("Test message")
  print(response)
except Exception as e:
  print(f"Error details: {e}")
  
  # Wrap for RAGAS
ragas_llm = LangchainLLMWrapper(generator_llm)
ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

  # Use in RAGAS evaluation
#   results = evaluate(
#       dataset=your_dataset,
#       metrics=[faithfulness, answer_relevancy],
#       llm=ragas_llm,
#       embeddings=ragas_embeddings
#   )

content="Got it—your test message came through successfully! Let me know if there's anything I can help you with." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 69, 'total_tokens': 109, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'openai/gpt-oss-120b', 'id': 'chatcmpl-hmfdt2wz9f4ef526q6bc67', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None} id='run--49ce95e8-3594-4ce5-8679-325a8ce63582-0' usage_metadata={'input_tokens': 69, 'output_tokens': 40, 'total_tokens': 109, 'input_token_details': {}, 'output_token_details': {}}


In [25]:
from ragas.testset.graph import KnowledgeGraph
kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

In [17]:
from ragas.testset.graph import Node, NodeType

for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 64, relationships: 0)

In [ ]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = ragas_llm
embedding_model = ragas_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

In [19]:
kg.save("graph_created_oss120.json")
usecase_data_kg=KnowledgeGraph.load("graph_created_oss120.json")
usecase_data_kg

KnowledgeGraph(nodes: 85, relationships: 744)

In [ ]:
# from ragas.testset import TestsetGenerator
# from ragas.testset.transforms import default_transforms

# generator = TestsetGenerator(llm=generator_llm, embedding_model=ragas_embeddings)
# dataset = generator.generate_with_langchain_docs(documents=docs, 
#     testset_size=10,
#     transforms=default_transforms,
#     transforms_llm=ragas_llm,
#     transforms_embedding_model=ragas_embeddings,
#     with_debugging_logs=True
#     )

In [26]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=ragas_llm, embedding_model=ragas_embeddings)

In [ ]:
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

In [29]:
generator_llm.invoke("What is the capital of Germany")

AIMessage(content='The capital of Germany is **Berlin**.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 85, 'total_tokens': 113, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'openai/gpt-oss-120b', 'id': 'chatcmpl-fdd63bpv4olw0dj38lcq1a', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--5dad7513-3ecb-4a19-a71e-e521d13221c0-0', usage_metadata={'input_tokens': 85, 'output_tokens': 28, 'total_tokens': 113, 'input_token_details': {}, 'output_token_details': {}})

In [ ]:
generator_llm.invoke("Explain quantum mechanics for a six year old.")

In [ ]:
dataset2 = generator.generate_with_langchain_docs(docs, testset_size=10)

In [36]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,What did Reuters report about US adult usage o...,[Introduction ChatGPT launched in November 202...,Reuters (2025) reports that 28% of US adults u...,single_hop_specifc_query_synthesizer
1,How many weekly messages does ChatGPT handle a...,[Introduction ChatGPT launched in November 202...,ChatGPT processes about 18 billion messages ea...,single_hop_specifc_query_synthesizer
2,What percentage of work‑related ChatGPT messag...,[Variation by Occupation Figure 23 presents va...,"According to the study, 32% of the work‑relate...",single_hop_specifc_query_synthesizer
3,"According to the analysis, what proportion of ...",[Variation by Occupation Figure 23 presents va...,The study reports that 47% of the work‑related...,single_hop_specifc_query_synthesizer
4,How many usres were using ChatGPT weekly by Ju...,[Conclusion This paper studies the rapid growt...,"By July 2025, ChatGPT had been used weekly by ...",single_hop_specifc_query_synthesizer
5,What does the paper say about Brynjolfsson's e...,[Conclusion This paper studies the rapid growt...,The paper cites Collis and Brynjolfsson (2025)...,single_hop_specifc_query_synthesizer
6,How does the study report regression adjustmen...,[<1-hop>\n\nVariation by Occupation Figure 23 ...,"The paper explains that, after controlling for...",multi_hop_abstract_query_synthesizer
7,How do the observed differences in work‑relate...,[<1-hop>\n\nVariation by Occupation Figure 23 ...,The data show that users in highly paid profes...,multi_hop_abstract_query_synthesizer
8,What are the econmic impcat and welfare gains ...,[<1-hop>\n\nVariation by Occupation Figure 23 ...,The study finds that users in highly‑paid prof...,multi_hop_abstract_query_synthesizer
9,How many percent of work‑related ChatGPT messa...,[<1-hop>\n\nVariation by Occupation Figure 23 ...,The study reports that 47% of the work‑related...,multi_hop_abstract_query_synthesizer


In [38]:
  # LM Studio uses OpenAI-compatible API format
fast_generator_llm = ChatOpenAI(
      base_url="http://192.168.1.157:1234/v1",  # Replace with your server 
      api_key="lmstudio",  # LM Studio doesn't require real API key
      model="openai/gpt-oss-20b",  # Optional: specify model name
      temperature=0.7,
      verbose=True 
  )

try:
  response = fast_generator_llm.invoke("Test message")
  print(response)
except Exception as e:
  print(f"Error details: {e}")

content='Got it! How can I help you today?' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 69, 'total_tokens': 101, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'openai/gpt-oss-20b', 'id': 'chatcmpl-hw5sr4lvy3j4reguta20q9', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None} id='run--93514c0b-82d5-448f-8ca8-4e0248cc8f31-0' usage_metadata={'input_tokens': 69, 'output_tokens': 32, 'total_tokens': 101, 'input_token_details': {}, 'output_token_details': {}}


In [39]:
fast_ragas_llm = LangchainLLMWrapper(fast_generator_llm)

In [ ]:
fast_generator = TestsetGenerator(llm=fast_ragas_llm, embedding_model=ragas_embeddings)
dataset = fast_generator.generate_with_langchain_docs(docs, testset_size=10)

In [41]:
dataset.to_pandas()


,user_input,reference_contexts,reference,synthesizer_name
0,Wht is the role of OepnAI in the diffusion of ...,[Introduction ChatGPT launched in November 202...,"OpenAI launched ChatGPT in November 2022, and ...",single_hop_specifc_query_synthesizer
1,what does kulveit et al say about ai impact on...,[Introduction ChatGPT launched in November 202...,"The paper cites Kulveit et al., 2025 as one of...",single_hop_specifc_query_synthesizer
2,how many people use chatgpt and how fast it sp...,[Introduction ChatGPT launched in November 202...,ChatGPT launched in November 2022. By July 202...,single_hop_specifc_query_synthesizer
3,How prevalent is Writing as a ChatGPT use case?,[Conclusion This paper studies the rapid growt...,"Writing is the most common work use, accountin...",single_hop_specifc_query_synthesizer
4,What does the research say about how professio...,[Conclusion This paper studies the rapid growt...,The study finds that Writing is the most commo...,single_hop_specifc_query_synthesizer
5,Wht is the proportion of ChatGPT messages that...,[Conclusion This paper studies the rapid growt...,The study finds that Seeking Information is on...,single_hop_specifc_query_synthesizer
6,Wht is the rapid adption and usage statisics o...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"The paper reports that by July 2025, ChatGPT h...",multi_hop_abstract_query_synthesizer
7,What rapid adoption statistics and usage patte...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"By July 2025, ChatGPT had been used weekly by ...",multi_hop_abstract_query_synthesizer
8,Wht is the rapid adption and usage statisics o...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"The paper reports that by July 2025, ChatGPT h...",multi_hop_abstract_query_synthesizer
9,Wht how does the rapid adption and diffusion o...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"The context shows that by July 2025, ChatGPT w...",multi_hop_abstract_query_synthesizer


In [46]:
from ragas.testset.graph import KnowledgeGraph
kg_fast = KnowledgeGraph()
kg_fast

KnowledgeGraph(nodes: 0, relationships: 0)

In [47]:

for doc in docs:
    kg_fast.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg_fast

KnowledgeGraph(nodes: 64, relationships: 0)

In [48]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = fast_ragas_llm
embedding_model = ragas_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg_fast, default_transforms)
kg_fast

Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/40 [00:00<?, ?it/s]

Property 'summary' already exists in node 'd34d5d'. Skipping!
Property 'summary' already exists in node 'f73d44'. Skipping!
Property 'summary' already exists in node 'b0fa7f'. Skipping!
Property 'summary' already exists in node '0c1dfe'. Skipping!
Property 'summary' already exists in node '202348'. Skipping!
Property 'summary' already exists in node '584a4b'. Skipping!
Property 'summary' already exists in node '8b043a'. Skipping!
Property 'summary' already exists in node '8e51e3'. Skipping!
Property 'summary' already exists in node '7ec953'. Skipping!
Property 'summary' already exists in node '06e3d5'. Skipping!
Property 'summary' already exists in node 'ba8ac1'. Skipping!
Property 'summary' already exists in node '762dda'. Skipping!
Property 'summary' already exists in node '69e04c'. Skipping!
Property 'summary' already exists in node '9fecc1'. Skipping!
Property 'summary' already exists in node 'e62a55'. Skipping!
Property 'summary' already exists in node '85b166'. Skipping!
Property

Applying CustomNodeFilter:   0%|          | 0/4 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/44 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node 'd34d5d'. Skipping!
Property 'summary_embedding' already exists in node 'f73d44'. Skipping!
Property 'summary_embedding' already exists in node 'b0fa7f'. Skipping!
Property 'summary_embedding' already exists in node '0c1dfe'. Skipping!
Property 'summary_embedding' already exists in node '202348'. Skipping!
Property 'summary_embedding' already exists in node '584a4b'. Skipping!
Property 'summary_embedding' already exists in node '8b043a'. Skipping!
Property 'summary_embedding' already exists in node '8e51e3'. Skipping!
Property 'summary_embedding' already exists in node '7ec953'. Skipping!
Property 'summary_embedding' already exists in node '06e3d5'. Skipping!
Property 'summary_embedding' already exists in node 'ba8ac1'. Skipping!
Property 'summary_embedding' already exists in node '762dda'. Skipping!
Property 'summary_embedding' already exists in node '69e04c'. Skipping!
Property 'summary_embedding' already exists in node '9fecc1'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 85, relationships: 782)

In [50]:
kg_fast.save("graph_created_oss20b.json")
fast_usecase_data_kg=KnowledgeGraph.load("graph_created_oss20b.json")
fast_usecase_data_kg

KnowledgeGraph(nodes: 85, relationships: 782)

In [61]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=fast_ragas_llm, embedding_model=ragas_embeddings, knowledge_graph=fast_usecase_data_kg)

In [62]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=fast_ragas_llm), 0.1),
        (MultiHopAbstractQuerySynthesizer(llm=fast_ragas_llm), 0.45),
        (MultiHopSpecificQuerySynthesizer(llm=fast_ragas_llm), 0.45),
]

In [ ]:
big_testset = generator.generate(testset_size=10, query_distribution=query_distribution)
big_testset.to_pandas()